# 服务器运行 —— PPO(proactive Rail Route)训练矩阵 + 断点续训 + 进度

**目的(D59/D60)**:在 **proactive wide 环境**上训 maskable PPO。**选路 KL 锚到规则参照**(schedule 计划路 + lookup 兜底,按 RR),
**热启动 = full-panel BC 整网**(D9 最优 gnn),**kind 时机放开由 RL 自学**,reward=per_train(D58)。

**硬件选择(实测决定)**:net 只有 hidden=64、rollout batch=1,GPU 前向 73% 是 kernel 启动开销(D32,换 A100 也一样)——
**用 no-GPU / 72 核**,吞吐来自把消融矩阵 N 路单核并发,而非单 run 加速。每 run 强制单核(线程=1)。

**断点续训(服务器单次 12h 上限)**:两层——矩阵级幂等(已完成 config 重跑自动跳过)+ run 级 `--resume`(从 `<tag>_last.pt` 续)。
**到点被杀后,重跑启动格即接着跑完。**

**纪律**:test 全程不碰(D43,headline 留到最后一次性报);**D45 作废**——先直接跑看 EV 能否自恢复,不预设强锚补丁。


## 设置(服务器前置 + 核数)


In [ ]:
import sys, os
venv_site = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/lib/python3.11/site-packages'
if os.path.isdir(venv_site) and venv_site not in sys.path:
    sys.path.insert(0, venv_site)
%cd /rds/homes/h/hxn886/ondemand/final_project

PROJ = '/rds/homes/h/hxn886/ondemand/final_project'
PY   = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/bin/python'
PREFIX = (f'cd {PROJ} && env PYTHONUNBUFFERED=1 PYTHONPATH=. '
          f'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MPLBACKEND=Agg {PY}')
N_WORKERS = min(70, (os.cpu_count() or 8))   # no-GPU 节点 72 核 → 留 ~2 核给系统
print('cpu_count =', os.cpu_count(), '| N_WORKERS =', N_WORKERS)
print('OK. shell 用  !{PREFIX} <script> <args>;  矩阵用 rl_matrix')


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available(), '(本流程不需要 GPU,CPU 72 核即可)')
print('torch threads (子进程内会被强制成 1):', torch.get_num_threads())


## 前置:full-panel BC 权重(热启动源)
矩阵热启动读 `artifacts/dataset/bc_fullpanel_<enc>.pt`(由 `bc_fullpanel_server.ipynb` 训出)。先确认存在。


In [ ]:
from pathlib import Path
for enc in ['gnn','mlp','deepsets','settransformer']:
    p = Path('artifacts/dataset') / f'bc_fullpanel_{enc}.pt'
    print(f'{enc:<14}', 'OK' if p.exists() else 'MISSING', p)


## 0) Smoke —— 1 个 config 小样本(先看热启动体检 + EV,别急着全量)
关键先看两件事:①**热启动体检**动作构成是否正常(WAIT/set/cancel,cancel 不该异常高);②几轮内 **EV 是否起来**(EV≈0 说明 critic 只在编码步序=优势是位置噪声)。这两点决定要不要调,再上全量。


In [ ]:
!{PREFIX} scripts/13_train_ppo.py --encoder gnn \
    --bc-ckpt artifacts/dataset/bc_fullpanel_gnn.pt \
    --kl-anchor schedule --kl-beta 0.1 --seed 0 \
    --run-tag smoke_gnn_schedule --reserve-window wide \
    --iters 6 --critic-warmup-iters 2 --steps-per-iter 512 \
    --days 4 --eval-days 2 --eval-every 2 --early-stop-patience 0


## ★ 单种子 curriculum 训练(当前主线)

**一个种子、一个 config、curriculum 由易到难**(D63):按每天车流量分 4 档累加喂天(≤185→≤286→≤300→全部 = 9→33→48→64 天),当前档 **train 完成率≥0.9 或满 tier-max-iters** 就升档,**最后跑满高峰**(守 D27)。

- **实时进度**:逐 iter 打印(含档位、EV、选路KL、早停),每评估周期打印 **train 完成率 + val 完成率 + 档位**(两套都看得到)。
- **断点续训**:`--run-tag` + `--resume`;12h 被杀后重跑本格即从断点续(连档位一起恢复)。
- 环境/奖励/网络一律不动,只改喂哪些天。


In [ ]:
# 单种子 curriculum 训练(重跑本格 = 断点续训)。iters 可调;early-stop 只在最终档生效。
!{PREFIX} scripts/13_train_ppo.py --encoder gnn \
    --bc-ckpt artifacts/dataset/bc_fullpanel_gnn.pt \
    --kl-anchor schedule --kl-beta 0.1 --seed 0 --reserve-window wide \
    --curriculum --curriculum-tiers 185,286,300,999999  --promote-at 0.9 \
    --train-eval-days 6 --iters 80 --steps-per-iter 2048 --eval-every 5 \
    --early-stop-patience 6 --run-tag cur_gnn_schedule_s0 --resume


### 结果:train / val 完成率随 iter(两套都记录)
跑完读 `<tag>.json`,跑一半读 `<tag>_last.pt`;打印 train 完成率 + val 完成率 + 档位 表 + 曲线。


In [ ]:
import json, torch
from pathlib import Path
tag='cur_gnn_schedule_s0'; rep=Path('artifacts/reports')
evals=None; best=None
if (rep/f'{tag}.json').exists():
    d=json.loads((rep/f'{tag}.json').read_text()); evals=d['evals']; best=d['best']
elif (rep/f'{tag}_last.pt').exists():
    st=torch.load(rep/f'{tag}_last.pt', map_location='cpu', weights_only=False); evals=st['evals']; best=st['best']
if not evals:
    print('还没有评估结果(run 还没到第一个 eval 周期)')
else:
    print(f"{'iter':>4} {'tier':>4} {'train完成':>9} {'val完成':>8} {'val_reward':>11}  WAIT/set/cxl")
    for e in evals:
        tc=e.get('train_completion'); tcs=(f'{tc:.3f}' if tc is not None else '  -  ')
        print(f"{e['iter']:>4} {e.get('tier',0)+1:>4} {tcs:>9} {e['eval_completion_rate']:>8.3f} "
              f"{e['eval_episode_reward']:>11.0f}  {e['eval_frac_wait']:.0%}/{e['eval_frac_set']:.0%}/{e['eval_frac_cancel']:.0%}")
    print(f"\n最佳(val): it{best['iter']} 完成率 {best['completion']:.3f} reward {best['reward']:.1f}")
    try:
        import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
        its=[e['iter'] for e in evals]
        plt.figure(figsize=(7,3))
        plt.plot(its,[e['eval_completion_rate'] for e in evals],'-o',label='val')
        plt.plot(its,[e.get('train_completion') for e in evals],'-s',label='train(cur tier)')
        plt.axhline(1.0,ls='--',c='g',lw=0.8,label='signaller=1.0'); plt.ylim(0,1.05)
        plt.xlabel('iter'); plt.ylabel('completion'); plt.legend(); plt.title(tag); plt.tight_layout()
        plt.savefig(rep/f'{tag}_curve.png',dpi=110); plt.show(); print('saved', rep/f'{tag}_curve.png')
    except Exception as ex: print('plot skipped:', ex)


## (可选,之后再铺)主扫 A —— 宽窗(wide) gnn 全矩阵
可重跑本格=断点续跑(已完成跳过,未完成 `--resume`)。21 个 run。逐 run 明细 `artifacts/reports/rl_logs/<tag>.log`。


In [ ]:
from derbysim.learn.rl_matrix import default_grid, run_rl_matrix, summarize_rl_matrix
grid_wide = default_grid(encoders=['gnn'])         # 21 runs
print('wide runs:', len(grid_wide))
run_rl_matrix(grid_wide, py=PY, n_workers=N_WORKERS,
              iters=40, steps_per_iter=2048, eval_every=5,
              reserve_window='wide', early_stop_patience=6)


In [ ]:
# (可选)只看目前已完成的排名; 完整对比见 §3 联合汇总
summarize_rl_matrix()


## 2) 主扫 B —— **窄窗(narrow)**(可选):上一轮 narrow 爬不过热启动(best≈0.501),wide 更高
先看 §1;想对照再跑。tag 加 `_nw`。


In [ ]:
grid_narrow = default_grid(encoders=['gnn'])
for c in grid_narrow: c['tag'] = c['tag'] + '_nw'
print('narrow runs:', len(grid_narrow))
run_rl_matrix(grid_narrow, py=PY, n_workers=N_WORKERS,
              iters=40, steps_per_iter=2048, eval_every=5,
              reserve_window='narrow', early_stop_patience=6)


## 3) 联合汇总(wide vs narrow 并列,按 val best reward 排名 + lookup/人类参照线)


In [ ]:
summarize_rl_matrix()   # 含 window 列: 直接看 wide 与 narrow 哪个起来、是否压过 lookup 靶子


## 4) 编码器消融(可选,在 §3 选出的最佳 window/anchor/β 上扩到 4 编码器)
先看 §3 联合汇总定最佳 **window**(wide/narrow)、anchor、β,再放开编码器做 D9 消融。改下面 3 个变量后跑。


In [ ]:
BEST_WINDOW, BEST_ANCHOR, BEST_BETA = 'wide', 'schedule', 0.1   # ← 按 §3 结果改
grid4 = default_grid(encoders=['mlp','deepsets','settransformer','gnn'],
                     anchors=[BEST_ANCHOR], betas=[BEST_BETA], seeds=[0,1,2])
if BEST_WINDOW=='narrow':
    for c in grid4: c['tag'] += '_nw'
print('runs:', len(grid4))
run_rl_matrix(grid4, py=PY, n_workers=N_WORKERS, iters=40, steps_per_iter=2048,
              eval_every=5, reserve_window=BEST_WINDOW, early_stop_patience=6)
summarize_rl_matrix()


## 进度监控(启动格在跑时,另开终端跑;或中断后跑本格看状态)
启动格阻塞期间同一内核跑不了别的格 —— 要**实时**看逐 iter,请在终端:
`tail -f artifacts/reports/rl_logs/ppo_gnn_schedule_b0.1_s0.log`。本格给一个**快照**:哪些 config 完成/在跑 + 各自最新一行。


In [ ]:
from pathlib import Path
rep, logs = Path('artifacts/reports'), Path('artifacts/reports/rl_logs')
done = {p.stem for p in rep.glob('ppo_*.json') if 'summary' not in p.name}
print('已完成(写了最终 .json):', len(done))
for lg in sorted(logs.glob('ppo_*.log')):
    tag = lg.stem
    lines = [l for l in lg.read_text(errors='ignore').splitlines() if l.strip()]
    last = lines[-1][:120] if lines else '(空)'
    print(f"{'✅' if tag in done else '…'} {tag:<34} {last}")


## 说明 / 交接
- **续训**:任一格中断/12h 被杀后,**重跑同一格**即可:已完成 config 跳过,未完成从 `<tag>_last.pt` 续。
- **产物**:每 run → `artifacts/reports/<tag>.json`(曲线/最佳/基线)+ `<tag>_best.pt`(最佳权重)+ `<tag>_last.pt`(续训全状态);日志 `rl_logs/<tag>.log`;汇总 `rl_matrix_summary_*.json`。
- **诊断先看**:热启动体检动作构成 → EV 是否起来 → 选路KL/早停率 → val reward 是否压过 lookup 靶子。**D45 作废**:先看 EV 自恢复,不预设补丁。
- **test 不碰**(D43):以上全在 train/val;headline 留到最后一次性在 test 段报(单独步骤)。
